# NB08 — 4-Model Bake-off (Appendix C)

Reproduces the NB04 model-selection step that produced
`event_study_best.pkl`. Runs four classifier families
(logistic regression, SVM, Random Forest, XGBoost) plus a
prior-strategy DummyClassifier baseline, with TimeSeriesSplit CV
and GridSearchCV on the train set, scored on the held-out 2023
validation set.

The Random Forest row should reproduce the pickled
`cv_auc_tuned = 0.518` and `val_auc_tuned = 0.601` numbers — that's
the sanity check that our pipeline matches NB04's exactly. The other
three families (logreg / SVM / XGB) are independently re-derived
here, not lifted from NB04.

**Why this notebook exists.** The paper's appendix C reports the
4-model comparison that justified picking RF for tuning. Without
this notebook the comparison numbers would be retained only in
NB04, which mixes bake-off with Optuna and final refit. NB08
isolates the bake-off so the comparison table is reproducible
in one cell.

**Sections**
1. Load data and build feature matrix
2. Run the bake-off
3. Sanity check the RF row against the pickle
4. Inspect best hyperparameters
5. Outputs

## 1. Load data and build feature matrix

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..').resolve()))
import config
from src.llm_agent.data_loader import load_sessions, load_prices, load_llm_signals
from src.llm_agent.features    import build_feature_matrix
from src.llm_agent.bakeoff     import run_bakeoff

# Build the same feature matrix NB04 uses (arm 'C' = 22 features)
sessions = load_sessions()
prices   = load_prices()
sigs     = load_llm_signals()
base = build_feature_matrix(sessions, prices, sigs, arm='C')

# Attach the NB04 target
car = pd.read_parquet(config.CLEAN_DIR / 'car_labels.parquet')
base = base.merge(car[['session_id','car_2_11','sigma_e']],
                  on='session_id', how='left')
base['y'] = (base['car_2_11'] > 0).astype('Int64')

# Pin feature order to the pkl so the RF row reproduces exactly
bundle = joblib.load(config.MODELS_DIR / 'event_study_best.pkl')
feats = bundle['features']
print(f'using {len(feats)} features in pinned order')

using 22 features in pinned order


## 2. Run the bake-off

NB04 ran logistic regression, linear/RBF SVM, RF, and XGBoost (when
available). We add the DummyClassifier(strategy='prior') as a
sanity baseline — its CV AUC should be 0.5 by construction.

Hyperparameter grids are the ones in `src/llm_agent/models.py`,
unchanged from NB04. Tail filter applies to train + val to match
NB04's regime; test sets are not produced here.

In [2]:
summary = run_bakeoff(
    base=base,
    feats=feats,
    train_years=config.TRAIN_YEARS,
    val_years=config.VAL_YEARS,
    n_splits=5,
)

display_cols = ['model', 'n_train', 'n_val',
                'cv_auc_mean', 'cv_auc_std', 'val_auc',
                'val_top_decile_precision']
summary[display_cols].round(4)

  fitting baseline...
  fitting logreg...
  fitting svm...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to it

  fitting rf...


  fitting xgb...


,model,n_train,n_val,cv_auc_mean,cv_auc_std,val_auc,val_top_decile_precision
0,baseline,537,108,0.5000,0.0000,0.5000,0.4
1,logreg,537,108,0.5093,0.0579,0.5552,0.6
2,svm,537,108,0.5262,0.0701,0.4571,0.3
3,rf,537,108,0.5451,0.0728,0.5063,0.5
4,xgb,537,108,0.5501,0.0775,0.5569,0.5


## 3. Sanity check vs the pickle

The pickle stores `cv_auc_tuned = 0.518, val_auc_tuned = 0.601`. These
are **not** comparable to our bake-off RF row directly, because NB04
runs a two-step procedure:

  1. **Bake-off** with the GridSearchCV grid in `models.py`
     (e.g. RF: `max_depth ∈ {3, 6, None}`, `min_samples_leaf ∈ {1, 5}`).
     This is what NB08 reproduces above. It selects the *family*.
  2. **Optuna fine-tune** of the chosen family on a wider grid
     (`max_depth ∈ [3, 25]`, `min_samples_leaf ∈ [5, 50]`,
     `max_features ∈ [0.3, 1.0]`), producing the pkl's
     `best_params = {max_depth: 19, min_samples_leaf: 20,
     max_features: 0.8}`.

So `cv_auc_tuned = 0.518` is the Optuna-tuned RF's CV score, not the
bake-off RF's. To verify the pickle, we refit a fresh RF with the
exact pkl params and re-score.

In [3]:
# Refit RF with the EXACT pkl params (Optuna output, not the bake-off grid)
from sklearn.ensemble       import RandomForestClassifier
from sklearn.preprocessing  import StandardScaler
from sklearn.pipeline       import Pipeline
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics        import roc_auc_score

# Get train / val with the same regime
from src.llm_agent.bakeoff import _split_by_period
train, val = _split_by_period(base, config.TRAIN_YEARS, config.VAL_YEARS, feats)
X_tr, y_tr = train[feats].astype(float).values, train['y'].astype(int).values
X_vl, y_vl = val[feats].astype(float).values,   val['y'].astype(int).values

# Plug in pkl's exact params
pkl_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=500,
        random_state=0,
        n_jobs=-1,
        **bundle['best_params'],
    )),
])

cv = TimeSeriesSplit(n_splits=5)
cv_scores = cross_val_score(pkl_pipe, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)
pkl_pipe.fit(X_tr, y_tr)
val_auc = roc_auc_score(y_vl, pkl_pipe.predict_proba(X_vl)[:, 1])

print(f'Reproduced from pkl best_params:')
print(f'  CV AUC  = {cv_scores.mean():.4f}   (pkl cv_auc_tuned  = {bundle["cv_auc_tuned"]:.4f})')
print(f'  Val AUC = {val_auc:.4f}   (pkl val_auc_tuned = {bundle["val_auc_tuned"]:.4f})')
print()
print(f'  CV folds: {[f"{s:.3f}" for s in cv_scores]}')

Reproduced from pkl best_params:
  CV AUC  = 0.5182   (pkl cv_auc_tuned  = 0.5180)
  Val AUC = 0.6012   (pkl val_auc_tuned = 0.6015)

  CV folds: ['0.476', '0.458', '0.530', '0.548', '0.580']


## 4. Best hyperparameters per family

Reported alongside the AUC numbers. NB04 selected RF on val AUC ×
CV-stability (lowest std), then ran Optuna on RF only.

In [4]:
for _, row in summary.iterrows():
    if row.best_params:
        print(f'{row.model:>10s}: {row.best_params}')
    else:
        print(f'{row.model:>10s}: (no grid)')

  baseline: (no grid)
    logreg: {'clf__C': 10.0, 'clf__penalty': 'l2'}
       svm: {'clf__C': 10.0, 'clf__gamma': 'scale', 'clf__kernel': 'linear'}
        rf: {'clf__max_depth': None, 'clf__min_samples_leaf': 1}
       xgb: {'clf__max_depth': 5, 'clf__subsample': 1.0}


## 5. Outputs

In [5]:
out_dir = config.PROJECT_ROOT / 'output' / 'tables'
out_dir.mkdir(parents=True, exist_ok=True)

# Save the summary minus the dict column (CSV-unfriendly)
out = summary.copy()
out['best_params'] = out['best_params'].apply(str)
out.to_csv(out_dir / 'bakeoff_summary.csv', index=False)
print(f'Saved: {out_dir.relative_to(config.PROJECT_ROOT)}/bakeoff_summary.csv')

Saved: output/tables/bakeoff_summary.csv
